# Baseline Experiments: Claim Detection and Typology Classification

This notebook establishes baseline performance for the two classification tasks
in our pipeline using FastText embeddings (crawl-300d-2M-subword) paired with
three models: Logistic Regression, Gaussian Naive Bayes, and a BiLSTM.

These baselines are reported in Table 3 of the paper:
*Claim Detection and Span Extraction in Reddit Health Discussions*

**Tasks evaluated:**
- Binary claim detection (full corpus, 1,215 posts)
- Claim typology classification: explicit vs. implicit (claim-positive posts only)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install gensim scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 24.6 MB/s eta 0:00:00


In [21]:
import pandas as pd
import numpy as np
import re

from gensim.models import KeyedVectors
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm

## Setup

Mount Google Drive and install dependencies.
FastText embeddings (`crawl-300d-2M-subword`) must be placed in:
`/content/drive/MyDrive/claim_detection/`

We use TF-IDF weights to compute weighted sentence embeddings from FastText vectors,
which gives better signal than simple averaging for short, noisy Reddit posts.

In [4]:
FASTTEXT_PATH = "/content/drive/MyDrive/claim_detection/crawl-300d-2M-subword.vec"

print("Loading FastText...")
fasttext = KeyedVectors.load_word2vec_format(FASTTEXT_PATH)
print("Loaded FastText")

Loading FastText...
Loaded FastText


## Data Loading

Load the annotated Reddit corpus from `labels_v5.csv`.

Two subsets are prepared:
- Full corpus (1,215 posts) for binary claim detection
- Claim-positive posts only for typology classification (explicit vs. implicit)

Labels:
- `claim`: 1 if post contains a self-medication claim, 0 otherwise
- `explicit`: 1 if claim is explicit, 0 if implicit (only defined for claim=1 posts)

In [5]:
DATA_PATH = "/content/drive/MyDrive/claim_detection/labels_v5.csv"

df = pd.read_csv(DATA_PATH)

texts = df["text"].astype(str).tolist()
claim_labels = df["claim"].astype(int).tolist()

df_claims = df[df["claim"] == 1].reset_index(drop=True)
texts_claim = df_claims["text"].astype(str).tolist()
type_labels = df_claims["explicit"].astype(int).tolist()

In [16]:
def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return text.split()

In [22]:
vectorizer = TfidfVectorizer(max_features=10000)
vectorizer.fit(texts)

idf_dict = dict(zip(vectorizer.get_feature_names_out(), vectorizer.idf_))

## Sentence Embedding

Each post is represented as a TF-IDF weighted average of its FastText word vectors.

Words not found in the FastText vocabulary receive a zero vector.
Posts with no recognized words receive a zero vector of dimension 300.
This weighted averaging is chosen over simple mean-pooling to downweight
high-frequency uninformative words.

In [23]:
def sentence_embedding(text):
    words = tokenize(text)

    vecs = []
    weights = []

    for w in words:
        if w in fasttext:
            vecs.append(fasttext[w])
            weights.append(idf_dict.get(w, 1.0))

    if len(vecs) == 0:
        return np.zeros(300)

    vecs = np.array(vecs)
    weights = np.array(weights).reshape(-1, 1)

    return np.sum(vecs * weights, axis=0) / np.sum(weights)

In [24]:
print("Encoding full dataset...")
X = np.array([sentence_embedding(t) for t in tqdm(texts)])

print("Encoding claim-only dataset...")
X_claim = np.array([sentence_embedding(t) for t in tqdm(texts_claim)])

Encoding full dataset...


100%|██████████| 1215/1215 [00:00<00:00, 1521.12it/s]


Encoding claim-only dataset...


100%|██████████| 878/878 [00:00<00:00, 1538.18it/s]


## Train/Test Split

In [25]:
X_train, X_val, y_train, y_val = train_test_split(
    X, claim_labels, test_size=0.2, random_state=42, stratify=claim_labels
)

Xc_train, Xc_val, yc_train, yc_val = train_test_split(
    X_claim, type_labels, test_size=0.2, random_state=42, stratify=type_labels
)

## Logistic Regression Baseline

Trained with `class_weight="balanced"` to partially compensate for the
explicit/implicit class imbalance in the typology task.

Reported metrics:
- Binary claim detection: F1 (positive class)
- Typology classification: per-class F1 for explicit and implicit

In [30]:
from sklearn.metrics import f1_score, classification_report

lr = LogisticRegression(max_iter=1000, class_weight="balanced")
lr.fit(X_train, y_train)

pred_claim = lr.predict(X_val)
f1_binary = f1_score(y_val, pred_claim)

lr_type = LogisticRegression(max_iter=1000, class_weight="balanced")
lr_type.fit(Xc_train, yc_train)

pred_type = lr_type.predict(Xc_val)
f1_type = f1_score(yc_val, pred_type, average="macro")

print("\nLogistic Regression")
print(f"Binary Claim F1: {f1_binary:.3f}")
print(f"Type Macro-F1: {f1_type:.3f}")

print("\nType Classification Report:")
print(classification_report(yc_val, pred_type, target_names=["implicit", "explicit"]))


Logistic Regression
Binary Claim F1: 0.795
Type Macro-F1: 0.497

Type Classification Report:
              precision    recall  f1-score   support

    implicit       0.13      0.50      0.21        16
    explicit       0.93      0.68      0.78       160

    accuracy                           0.66       176
   macro avg       0.53      0.59      0.50       176
weighted avg       0.86      0.66      0.73       176



## Gaussian Naive Bayes Baseline

Gaussian NB assumes features are normally distributed, which is a rough
approximation for dense embedding dimensions but provides a useful
non-parametric comparison. No class weighting is applied as GaussianNB
does not support it natively.

In [31]:
from sklearn.metrics import f1_score, classification_report

nb = GaussianNB()
nb.fit(X_train, y_train)

pred_claim_nb = nb.predict(X_val)
f1_binary_nb = f1_score(y_val, pred_claim_nb)

nb_type = GaussianNB()
nb_type.fit(Xc_train, yc_train)

pred_type_nb = nb_type.predict(Xc_val)
f1_type_nb = f1_score(yc_val, pred_type_nb, average="macro")

print("\nNaive Bayes (Gaussian)")
print(f"Binary Claim F1: {f1_binary_nb:.3f}")
print(f"Type Macro-F1: {f1_type_nb:.3f}")

print("\nType Classification Report:")
print(classification_report(yc_val, pred_type_nb, target_names=["implicit", "explicit"]))


Naive Bayes (Gaussian)
Binary Claim F1: 0.804
Type Macro-F1: 0.500

Type Classification Report:
              precision    recall  f1-score   support

    implicit       0.10      0.19      0.13        16
    explicit       0.91      0.83      0.87       160

    accuracy                           0.77       176
   macro avg       0.51      0.51      0.50       176
weighted avg       0.84      0.77      0.80       176



## BiLSTM Baseline

A bidirectional LSTM operating over sequences of FastText word embeddings
(truncated/padded to MAX_LEN=50 tokens).

Architecture:
- Input: sequence of 300-dim FastText vectors, length 50
- BiLSTM: hidden size 128, bidirectional -> 256-dim representation
- Linear head: 256 -> 1 (binary output)

Class imbalance is handled via `pos_weight` in BCEWithLogitsLoss,
computed as (negative samples) / (positive samples) per task.
Training: 5 epochs, Adam optimizer, lr=1e-3.

In [32]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import numpy as np


MAX_LEN = 50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def text_to_sequence(text):
    words = tokenize(text)

    vecs = []
    for w in words[:MAX_LEN]:
        if w in fasttext:
            vecs.append(fasttext[w])
        else:
            vecs.append(np.zeros(300))

    while len(vecs) < MAX_LEN:
        vecs.append(np.zeros(300))

    return np.array(vecs)


X_seq = np.array([text_to_sequence(t) for t in texts])
Xc_seq = np.array([text_to_sequence(t) for t in texts_claim])


X_train_s, X_val_s, y_train_s, y_val_s = train_test_split(
    X_seq, claim_labels, test_size=0.2, random_state=42, stratify=claim_labels
)

Xc_train_s, Xc_val_s, yc_train_s, yc_val_s = train_test_split(
    Xc_seq, type_labels, test_size=0.2, random_state=42, stratify=type_labels
)

# Dataset
class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_loader = DataLoader(SeqDataset(X_train_s, y_train_s), batch_size=32, shuffle=True)
val_loader = DataLoader(SeqDataset(X_val_s, y_val_s), batch_size=32)

train_loader_c = DataLoader(SeqDataset(Xc_train_s, yc_train_s), batch_size=32, shuffle=True)
val_loader_c = DataLoader(SeqDataset(Xc_val_s, yc_val_s), batch_size=32)


# Model
class BiLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=300,
            hidden_size=128,
            batch_first=True,
            bidirectional=True
        )
        self.fc = nn.Linear(256, 1)

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        h = torch.cat((h[0], h[1]), dim=1)
        return self.fc(h).squeeze(1)


# Training + Evaluation
def train_model(train_loader, val_loader, y_train, epochs=5):
    model = BiLSTM().to(device)

    # class imbalance handling
    pos_weight = torch.tensor(
        (len(y_train) - sum(y_train)) / sum(y_train),
        dtype=torch.float32
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    # training
    for epoch in range(epochs):
        model.train()
        for Xb, yb in train_loader:
            Xb = Xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(Xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

    # evaluation
    model.eval()
    preds, labels = [], []

    with torch.no_grad():
        for Xb, yb in val_loader:
            Xb = Xb.to(device)

            logits = model(Xb)
            p = (torch.sigmoid(logits) > 0.5).int().cpu().numpy()

            preds.extend(p)
            labels.extend(yb.numpy())

    macro_f1 = f1_score(labels, preds, average="macro")

    return macro_f1, labels, preds


# Run
f1_binary, y_true_b, y_pred_b = train_model(train_loader, val_loader, y_train_s)
f1_type, y_true_t, y_pred_t = train_model(train_loader_c, val_loader_c, yc_train_s)

print("\nBiLSTM")
print(f"Binary Claim F1: {f1_binary:.3f}")
print(f"Type Macro-F1: {f1_type:.3f}")

print("\nType Classification Report:")
print(classification_report(y_true_t, y_pred_t, target_names=["implicit", "explicit"]))


BiLSTM
Binary Claim F1: 0.668
Type Macro-F1: 0.487

Type Classification Report:
              precision    recall  f1-score   support

    implicit       0.09      0.19      0.12        16
    explicit       0.91      0.81      0.85       160

    accuracy                           0.75       176
   macro avg       0.50      0.50      0.49       176
weighted avg       0.83      0.75      0.79       176



## Summary of Baseline Results

| Model | Explicit F1 | Implicit F1 |
|---|---|---|
| Logistic Regression | 0.78 | 0.21 |
| Naive Bayes | 0.87 | 0.13 |
| BiLSTM | 0.85 | 0.12 |

**Key finding:** All three models perform adequately on explicit claims
but fail severely on implicit claims, confirming that static embeddings
and recurrent architectures cannot capture the cross-sentence pragmatic
context that implicit self-medication claims rely on.

This motivates the shift to a contextual BERT-mini architecture.